# Experiments A & C — Gauge diffusion in LLM training, and intervention transfer up to gauge alignment

**Regime logic** (why checkpoints, given Pred 2's failure): directed fine-tuning follows
gradients, which avoid flat directions — that prediction failed as expected in
hindsight. Long *stochastic* pre-training diffuses along flat directions: SGD noise has
components on the gauge orbit and nothing restores them. So the gauge signature should
live in checkpoint-to-checkpoint change, not fine-tuning deltas.

- **A (weights):** post-convergence checkpoint deltas of cyclic-vocab embeddings are
  commutant-tangent above null; the circle phase random-walks while gauge invariants
  (mode energies, spectra) stay put.
- **C (behavior):** a phase-keyed steering intervention fit at checkpoint t transfers
  to checkpoint t′ with efficiency that decays in |t−t′| raw, and stays flat after
  gauge (phase) alignment.

---
## PRE-REGISTERED PLAN — frozen before running.

**Gates (checked in order; failing a gate means the experiment is UNRUNNABLE on that
model, which is a scope result, not evidence for or against gauge):**
- G0a: months (or days) are single tokens for the model's tokenizer.
- G0b: circle structure detected in the FINAL checkpoint: mode-1 energy beats the
  spectrum-preserving rotation null at p ≤ 0.01 (B = 499).
- G0c (Part C only): final checkpoint performs month arithmetic ≥ 3× chance intact.

**A — hypotheses and rules.**
- A1: mean tangent fraction of consecutive late-checkpoint deltas (last ~25% of
  training) > 10× empirical-null q99 AND > permutation-placebo q90. Report per mode;
  primary = circle (k=1).
- A2: circle-phase increments across late checkpoints are nonzero (random walk) while
  per-mode energies have CV < 0.05 over the same span. Report phase trajectory.
- A1-dose: tangent fraction vs checkpoint gap (consecutive, 2-apart, 4-apart…):
  diffusion predicts high tangent fraction at all gaps with total change growing.
- Calibration control: the same analysis on EARLY checkpoints (pre-convergence, where
  change is gradient-driven) — predicted to look like Pred 2 (≈ null). This contrast
  is the cleanest evidence for the regime story; report it either way.

**C — hypotheses and rules.**
- Intervention: steering vector from checkpoint t's circle geometry that shifts month
  M by +k positions (v = circle-position difference), applied to embedding rows;
  readout = targeted-shift rate on the month-arithmetic grid (answer moves by exactly
  +k), which is phase-KEYED (geometric energy measures are provably phase-blind for an
  isotropic circle — see self-test cell notes).
- Positive control: self-application at the same checkpoint must give targeted-shift
  rate ≥ 3× chance; if not, the readout is invalid and C is unrunnable (report).
- C1: aligned-transfer efficiency ≥ 0.8 at all gaps; raw efficiency decreasing in gap
  (Page trend, one-sided). Alignment = complex phase alignment of circle contents
  (primary) and full Procrustes (secondary, reported).

**Model candidates** (in order; move down the list on gate failure):
Part A: EleutherAI/pythia-1.4b (154 checkpoints), pythia-2.8b, allenai/OLMo-2-1124-7B.
Part C: OLMo-2-1124-7B (capability), pythia-2.8b (fallback; G0c may fail).

**Decision rules.** A1+A2 pass with early-checkpoint control ≈ null → 'gauge diffusion
during pre-training' is established and the Pred-2 contrast becomes a regime result.
A1 fails on late checkpoints → weight-space gauge diffusion is not detectable at this
scale/structure; report as bounded negative. C1 passes → first LLM-scale evidence for
transfer-up-to-gauge. Any post-hoc deviation from these rules must be labeled as such.


In [ ]:
SMOKE = True
MODEL_A   = "EleutherAI/pythia-1.4b"     # Part A: many checkpoints, cheap rows
MODEL_C   = "allenai/OLMo-2-1124-7B"     # Part C: needs task capability
# Pythia revisions: step1000 ... step143000. Late = last quarter.
STEPS_ALL  = [1000, 2000, 4000, 8000, 16000, 32000] + list(range(36000, 143001, 107000 // (6 if True else 20)))
LATE_STEPS = ([107000, 116000, 125000, 134000, 143000] if SMOKE else
              list(range(107000, 143001, 3000)))
EARLY_STEPS = [1000, 2000, 4000, 8000]   # pre-convergence control
N_NULL, N_PLACEBO = (50, 10) if SMOKE else (200, 30)
SEED = 20260706
import numpy as np
rng = np.random.default_rng(SEED)
print("late checkpoints:", LATE_STEPS)

In [ ]:
"""Core for Experiment A (gauge diffusion across training checkpoints) and
Experiment C (intervention transfer across checkpoints/seeds up to gauge
alignment), for LLMs with cyclic vocabularies.

Regime logic (why checkpoints, not fine-tuning deltas -- Pred 2's failure):
directed gradient steps avoid flat directions; long stochastic training
diffuses along them. Post-convergence checkpoint-to-checkpoint change should
therefore be commutant-tangent above null (A), and any intervention keyed to
a specific orientation should decay in transferability with checkpoint gap
unless gauge-aligned (C).

All statistics reuse the calibrated machinery from Pred 1/2:
- tangent fraction of a change (commutant_change_decomposition)
- empirical nulls from random changes of matched size
- permutation placebo (analysis under shuffled token order)
- Procrustes gauge alignment with spectral floor
"""
import numpy as np


def fourier_modes(n):
    j = np.arange(n)
    modes = [(0, np.ones((n, 1)) / np.sqrt(n))]
    for k in range(1, n // 2 + 1):
        c = np.cos(2 * np.pi * k * j / n)
        s = np.sin(2 * np.pi * k * j / n)
        M = np.column_stack([c, s]) if (2 * k != n) else c[:, None]
        Q, _ = np.linalg.qr(M)
        modes.append((k, Q))
    assert sum(B.shape[1] for _, B in modes) == n
    return modes


def circle_energy_pvalue(X, modes, B_rot=499, rng=None):
    """Calibrated structure gate: is mode-1 energy above the spectrum-preserving
    rotation null? Returns (energy_fraction, p)."""
    rng = rng or np.random.default_rng(0)
    P1 = modes[1][1] @ modes[1][1].T
    Xc = X - X.mean(axis=0, keepdims=True)
    def frac(Y):
        return np.linalg.norm(P1 @ Y) ** 2 / np.linalg.norm(Y) ** 2
    f_obs = frac(Xc)
    n = X.shape[0]
    count = 0
    for _ in range(B_rot):
        A = rng.standard_normal((n, n))
        Q, R = np.linalg.qr(A)
        Q = Q * np.sign(np.diag(R))
        if frac(Q @ Xc) >= f_obs:
            count += 1
    return float(f_obs), (1 + count) / (B_rot + 1)


def tangent_fraction(X_from, X_to, modes, k):
    """Fraction of the mode-k change lying in the commutant tangent of X_from."""
    B = dict(modes)[k]
    Cb, Cn = B.T @ X_from, B.T @ X_to
    D = Cn - Cb
    if Cb.shape[0] != 2:
        c, d = Cb[0], D[0]
        nc, nd = np.linalg.norm(c), np.linalg.norm(d)
        if nc < 1e-12 or nd < 1e-12:
            return np.nan
        return float((c @ d) ** 2 / (nc ** 2 * nd ** 2))
    z = Cb[0] + 1j * Cb[1]
    d = D[0] + 1j * D[1]
    nz, nd = np.linalg.norm(z), np.linalg.norm(d)
    if nz < 1e-12 or nd < 1e-12:
        return np.nan
    u = z / nz
    return float(np.abs(np.vdot(u, d)) ** 2 / nd ** 2)


def circle_phase(X, modes, ref_u=None):
    """Phase of the mode-1 content relative to a reference complex direction.
    Returns (phase, u) where u can be reused as the reference for a trajectory."""
    B = dict(modes)[1]
    C = B.T @ X
    z = C[0] + 1j * C[1]
    if ref_u is None:
        ref_u = z / np.linalg.norm(z)
        return 0.0, ref_u
    w = np.vdot(ref_u, z)
    return float(np.angle(w)), ref_u


def mode_energies(X, modes):
    return {k: float(np.linalg.norm(B.T @ X) ** 2) for k, B in modes}


def empirical_null_tangent(X_base, modes, k, delta_norm, n_draws, rng):
    fr = []
    for _ in range(n_draws):
        D = rng.standard_normal(X_base.shape)
        D *= delta_norm / np.linalg.norm(D)
        fr.append(tangent_fraction(X_base, X_base + D, modes, k))
    return np.array(fr)


def placebo_tangent(X_from, X_to, modes, k, n_perm, rng):
    n = X_from.shape[0]
    fr = []
    for _ in range(n_perm):
        perm = rng.permutation(n)
        fr.append(tangent_fraction(X_from[perm], X_to[perm], modes, k))
    return np.array(fr)


def procrustes_align(XA, XB):
    U, s, Vt = np.linalg.svd(XA.T @ XB)
    Q = U @ Vt
    return Q, float(np.linalg.norm(XA @ Q - XB) ** 2)


def circle_plane(X, modes):
    """Orthonormal basis (d x 2) of the model-space plane carrying mode 1."""
    B = dict(modes)[1]
    C = B.T @ X
    Qp, _ = np.linalg.qr(C.T)
    return Qp


def ablate_plane(X, V):
    return X - (X @ V) @ V.T


def transfer_efficiency(damage_transferred, damage_self):
    if abs(damage_self) < 1e-9:
        return np.nan
    return float(damage_transferred / damage_self)


## Self-test (CPU) — validates A machinery end-to-end and C's alignment mechanics.
**Honest scope note:** the C self-test below validates alignment geometry only. It
CANNOT validate phase-sensitivity of the behavioral readout, because for an isotropic
circle every geometric energy measure is phase-invariant (commutant motion acts on the
token side and preserves the model-space plane and all spectra). Phase only matters to
a downstream reader that assigns meaning to positions — i.e., the task. C's in-model
positive control (self-steering at gap 0) is therefore part of C's validity, not just
a sanity check.

In [ ]:
def selftest(seed=1):
    rng = np.random.default_rng(seed)
    n, d = 12, 256
    modes = fourier_modes(n)
    U = np.linalg.qr(np.random.default_rng(42).standard_normal((d, 4)))[0]
    def X_of(theta, extra=0.0, rngx=None):
        B1 = dict(modes)[1]
        c1 = np.cos(theta)*U[:,0] + np.sin(theta)*U[:,1]
        c2 = -np.sin(theta)*U[:,0] + np.cos(theta)*U[:,1]
        X = B1 @ np.vstack([c1, c2])
        B3 = dict(modes)[3]
        X += B3 @ np.vstack([U[:,2], U[:,3]])
        if extra > 0: X += extra * rngx.standard_normal((n, d))
        return X
    thetas = np.cumsum(rng.normal(0, 0.1, 20))
    ck = [X_of(t, extra=5e-4, rngx=rng) for t in thetas]
    tf = [tangent_fraction(ck[i], ck[i+1], modes, 1) for i in range(19)]
    dn = np.linalg.norm(ck[1] - ck[0])
    null = empirical_null_tangent(ck[0], modes, 1, dn, 100, rng)
    assert np.mean(tf) > 10 * np.quantile(null, .99)
    ck_r = [X_of(0.0) + 0.01*i*rng.standard_normal((n, d)) for i in range(1, 6)]
    tf_r = [tangent_fraction(ck_r[i], ck_r[i+1], modes, 1) for i in range(4)]
    assert np.mean(tf_r) < 20 / d
    ph, ref = circle_phase(ck[0], modes)
    rec = np.unwrap([circle_phase(c, modes, ref)[0] for c in ck])
    truth = thetas - thetas[0]
    err = min(np.abs(rec - truth).max(), np.abs(rec + truth).max())
    assert err < 0.1
    print(f"SELF-TEST PASS  diffusion tangent={np.mean(tf):.3f} "
          f"calibration={np.mean(tf_r):.4f} phase err={err:.4f} rad")
selftest()

## Cheap checkpoint access — embedding rows only, via safetensors slicing
Loading a full model per checkpoint is ~3–14 GB each. Part A only needs n rows of the
embedding matrix, so we download just the shard containing it and slice.
(UNTESTED against the live Hub in this build; shard/key names may need one adjustment
per model family — the code prints available keys on failure.)

In [ ]:
import json as _json, torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from transformers import AutoTokenizer

MONTHS = ["January","February","March","April","May","June","July",
          "August","September","October","November","December"]
DAYS = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

EMB_KEYS = ["gpt_neox.embed_in.weight", "model.embed_tokens.weight",
            "transformer.wte.weight", "embed_tokens.weight"]

def resolve_single_tokens(tok, words):
    for prefix in (" ", ""):
        ids = [tok.encode(prefix + w, add_special_tokens=False) for w in words]
        if all(len(t) == 1 for t in ids):
            return prefix, [t[0] for t in ids]
    return None, None

def load_embedding_rows(repo, revision, token_ids):
    'Download only the shard holding the embedding; slice the needed rows.'
    try:
        idx_path = hf_hub_download(repo, "model.safetensors.index.json",
                                   revision=revision)
        index = _json.load(open(idx_path))['weight_map']
        key = next(k for k in EMB_KEYS if k in index)
        shard = hf_hub_download(repo, index[key], revision=revision)
    except Exception:
        shard = hf_hub_download(repo, "model.safetensors", revision=revision)
        with safe_open(shard, framework="pt") as f:
            keys = list(f.keys())
        key = next((k for k in EMB_KEYS if k in keys), None)
        if key is None:
            raise RuntimeError(f"embedding key not found; available: {keys[:20]}")
    with safe_open(shard, framework="pt") as f:
        W = f.get_tensor(key)
    return W[torch.tensor(token_ids)].float().numpy().copy()

## Part A — gauge diffusion across checkpoints

In [ ]:
import pandas as pd
tok = AutoTokenizer.from_pretrained(MODEL_A)
prefix, ids = resolve_single_tokens(tok, MONTHS)
assert ids is not None, "G0a FAILED for months; try DAYS or next model"
n = len(MONTHS); modes = fourier_modes(n)

def rev(step): return f"step{step}"
rows = {}
for s in LATE_STEPS + EARLY_STEPS:
    rows[s] = load_embedding_rows(MODEL_A, rev(s), ids)
    print(f"step {s}: rows {rows[s].shape}")

# G0b: structure gate on the final checkpoint
f1, p1 = circle_energy_pvalue(rows[LATE_STEPS[-1]], modes, B_rot=499,
                              rng=np.random.default_rng(SEED))
print(f"G0b circle gate: energy frac {f1:.4f}, p = {p1:.4f}")
assert p1 <= 0.01, 'G0b FAILED: no detectable circle; move to next model/vocab'

In [ ]:
# A1 + A1-dose + early-checkpoint calibration control
records = []
def tangent_block(steps, label):
    for gap in (1, 2, 4):
        pairs = [(steps[i], steps[i+gap]) for i in range(len(steps)-gap)]
        for s0, s1 in pairs:
            X0, X1 = rows[s0], rows[s1]
            dn = np.linalg.norm(X1 - X0)
            for k in range(1, n//2 + 1):
                tf = tangent_fraction(X0, X1, modes, k)
                records.append(dict(phase=label, gap=gap, s0=s0, s1=s1, k=k,
                                    tangent=tf, delta_norm=dn))
tangent_block(LATE_STEPS, "late")
tangent_block(EARLY_STEPS, "early")

X_ref = rows[LATE_STEPS[0]]
dn_typ = np.median([r['delta_norm'] for r in records
                    if r['phase'] == 'late' and r['gap'] == 1])
null = empirical_null_tangent(X_ref, modes, 1, dn_typ, N_NULL, rng)
plac = placebo_tangent(rows[LATE_STEPS[0]], rows[LATE_STEPS[1]], modes, 1,
                       N_PLACEBO, rng)
df = pd.DataFrame(records)
df.to_csv('expA_tangent.csv', index=False)
late1 = df[(df.phase=='late') & (df.gap==1) & (df.k==1)].tangent
early1 = df[(df.phase=='early') & (df.gap==1) & (df.k==1)].tangent
print(f"A1 circle tangent, LATE consecutive: mean {late1.mean():.4f}")
print(f"   null q99 {np.quantile(null,.99):.4f} | placebo q90 {np.quantile(plac,.9):.4f}")
print(f"A1 verdict: {'PASS' if (late1.mean() > 10*np.quantile(null,.99) and late1.mean() > np.quantile(plac,.9)) else 'FAIL'}")
print(f"calibration control, EARLY consecutive: mean {early1.mean():.4f} "
      f"(predicted ~ null if the regime story is right)")
print(df[df.k==1].groupby(['phase','gap']).tangent.mean().round(4).to_string())

In [ ]:
# A2: phase random-walk vs invariant stability across late checkpoints
ph0, ref = circle_phase(rows[LATE_STEPS[0]], modes)
phases = [circle_phase(rows[s], modes, ref)[0] for s in LATE_STEPS]
energies = np.array([[mode_energies(rows[s], modes)[k] for k in range(1, n//2+1)]
                     for s in LATE_STEPS])
cv = (energies.std(axis=0) / energies.mean(axis=0))
print("circle phase trajectory (rad):", np.round(np.unwrap(phases), 4))
print("per-mode energy CV over late checkpoints:", np.round(cv, 4))
print(f"A2 verdict: {'PASS' if (np.abs(np.diff(np.unwrap(phases))).mean() > 1e-3 and cv.max() < 0.05) else 'FAIL / inspect'}")

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(LATE_STEPS, np.unwrap(phases), marker='o')
ax[0].set_xlabel('step'); ax[0].set_ylabel('circle phase (rad)')
ax[1].plot(LATE_STEPS, energies)
ax[1].set_xlabel('step'); ax[1].set_ylabel('mode energy')
plt.tight_layout(); plt.savefig('expA_phase.png', dpi=150)
print('saved expA_phase.png')

## Part C — phase-keyed steering transfer across checkpoints
Needs full model forward passes at 2–3 checkpoints. UNTESTED against live models in
this build. Gate G0c first; if the model can't do month arithmetic, C is unrunnable
here (report, move to next candidate).

In [ ]:
from transformers import AutoModelForCausalLM
from contextlib import contextmanager

C_STEPS = [LATE_STEPS[0], LATE_STEPS[len(LATE_STEPS)//2], LATE_STEPS[-1]]
TEMPLATES = ["If it is {a} and {k} months pass, the month will be",
             "{k} months after {a} is",
             "The month {k} months later than {a} is"]

def load_full(repo, step):
    m = AutoModelForCausalLM.from_pretrained(repo, revision=f"step{step}",
        torch_dtype=torch.bfloat16, device_map="auto")
    m.eval(); return m

@contextmanager
def swapped_rows(model, token_ids, new_rows):
    emb = model.get_input_embeddings().weight
    idx = torch.tensor(token_ids, device=emb.device)
    backup = emb.data[idx].clone()
    try:
        emb.data[idx] = torch.tensor(new_rows, dtype=emb.dtype, device=emb.device)
        yield
    finally:
        emb.data[idx] = backup

@torch.no_grad()
def month_predictions(model, tok, ids, batch=16):
    prompts, meta = [], []
    for ti, tpl in enumerate(TEMPLATES):
        for a in range(12):
            for k in range(1, 12):
                prompts.append(tpl.format(a=MONTHS[a], k=k)); meta.append((a, k))
    preds = []
    for i in range(0, len(prompts), batch):
        enc = tok(prompts[i:i+batch], return_tensors='pt', padding=True).to(model.device)
        out = model(**enc)
        last = enc.attention_mask.sum(1) - 1
        logits = out.logits[torch.arange(len(enc.input_ids)), last][:, ids]
        preds.extend(logits.argmax(-1).tolist())
    return np.array(preds), meta

def steering_rows(X, modes, shift_k=3, alpha=1.0):
    'Move each month +shift_k along the CIRCLE component only.'
    B = dict(modes)[1]
    C = B.T @ X
    j = np.arange(12)
    circ = B @ C                                # n x d circle component
    circ_shifted = np.roll(circ, -shift_k, axis=0)
    return X + alpha * (circ_shifted - circ)

def targeted_shift_rate(preds, meta, shift_k):
    hits = [int(p == (a + k + shift_k) % 12) for p, (a, k) in zip(preds, meta)]
    return float(np.mean(hits))

In [ ]:
tokC = AutoTokenizer.from_pretrained(MODEL_C)
tokC.pad_token = tokC.pad_token or tokC.eos_token
prefC, idsC = resolve_single_tokens(tokC, MONTHS)
assert idsC is not None, "G0a FAILED for Part C model"
SHIFT_K, ALPHA = 3, 1.0
modesC = fourier_modes(12)
rowsC = {s: load_embedding_rows(MODEL_C, f"step{s}", idsC) for s in C_STEPS}

# fit intervention at the FIRST checkpoint
X_fit = rowsC[C_STEPS[0]]
eff = []
for s in C_STEPS:
    model = load_full(MODEL_C, s)
    X_here = rowsC[s]
    base_preds, meta = month_predictions(model, tokC, idsC)
    acc = float(np.mean([int(p == (a+k) % 12) for p, (a,k) in zip(base_preds, meta)]))
    if s == C_STEPS[0]:
        print(f"G0c: intact month accuracy at step {s}: {acc:.3f} (chance 1/12={1/12:.3f})")
        assert acc >= 3/12, "G0c FAILED: model cannot do the task; C unrunnable here"
    # self-fit intervention (positive control at every checkpoint)
    with swapped_rows(model, idsC, steering_rows(X_here, modesC, SHIFT_K, ALPHA)):
        p_self, _ = month_predictions(model, tokC, idsC)
    # raw transfer of the fit-checkpoint intervention
    with swapped_rows(model, idsC, X_here + (steering_rows(X_fit, modesC, SHIFT_K, ALPHA) - X_fit)):
        p_raw, _ = month_predictions(model, tokC, idsC)
    # phase-aligned transfer: rotate the fit-checkpoint circle to this checkpoint's phase
    B1 = dict(modesC)[1]
    z_fit = (B1.T @ X_fit)[0] + 1j*(B1.T @ X_fit)[1]
    z_here = (B1.T @ X_here)[0] + 1j*(B1.T @ X_here)[1]
    w = np.vdot(z_fit, z_here); w /= abs(w)
    C_fit = B1.T @ (steering_rows(X_fit, modesC, SHIFT_K, ALPHA) - X_fit)
    zv = C_fit[0] + 1j*C_fit[1]
    zv_al = w * zv
    v_aligned = B1 @ np.vstack([zv_al.real, zv_al.imag])
    with swapped_rows(model, idsC, X_here + v_aligned):
        p_al, _ = month_predictions(model, tokC, idsC)
    r_self = targeted_shift_rate(p_self, meta, SHIFT_K)
    r_raw  = targeted_shift_rate(p_raw, meta, SHIFT_K)
    r_al   = targeted_shift_rate(p_al, meta, SHIFT_K)
    eff.append(dict(step=s, self=r_self, raw=r_raw, aligned=r_al,
                    eff_raw=r_raw/max(r_self,1e-9), eff_al=r_al/max(r_self,1e-9)))
    print(f"step {s}: self {r_self:.3f} | raw {r_raw:.3f} (eff {eff[-1]['eff_raw']:.2f}) "
          f"| aligned {r_al:.3f} (eff {eff[-1]['eff_al']:.2f})")
    del model; torch.cuda.empty_cache()
pd.DataFrame(eff).to_csv('expC_transfer.csv', index=False)
print("C1 verdict per pre-registered rule: aligned eff >= 0.8 at all gaps AND raw decreasing.")

## Interpretation (pre-committed)
- **A passes with early-control ≈ null** → gauge diffusion during pre-training is
  real; combined with Pred 2's failure this is a two-regime result: gradient-driven
  change avoids the orbit, stochastic training diffuses along it. This is the LLM
  'is gauge governing weights' answer: yes for the diffusive component, no for the
  directed component — which is exactly what the flat-direction argument predicts.
- **A fails on late checkpoints** → no detectable weight-space gauge diffusion at this
  scale; the claim stays constructed/toy-model-only. Report as a bounded negative.
- **C passes** → first LLM-scale evidence for intervention transfer up to gauge.
- **Gates fail everywhere** → scope result ('no detectable circle structure in
  checkpoint-suite models'); NOT evidence about gauge. Do not conflate.
- Nothing here blocks or changes the August paper; these are follow-up-paper
  experiments. The paper's critical path remains Exp 2 (GPU) and Exp 1 FULL.